# 02 — RAG Pipeline
### Document Loading · Text Splitting · Embeddings · Vector Store · Retrieval · Generation


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1hpmLEFdpe5U7PSCZ416OgIGJycG5xPWs#scrollTo=buaknWr9Tlbv)
[![Python](https://img.shields.io/badge/Python-3.10%2B-blue)](https://python.org)
[![LangChain](https://img.shields.io/badge/LangChain-0.3.x-green)](https://python.langchain.com)
[![Gemini](https://img.shields.io/badge/LLM-Gemini%201.5%20Flash-orange)](https://aistudio.google.com)


### Copy the links from below badge to visit my pages


[![GitHub](https://img.shields.io/badge/GitHub-View_Profile-black?logo=github)](https://github.com/mtptisid)
<a href="https://www.linkedin.com/in/siddharamayya-mathapati" target="_blank">
  <img src="https://img.shields.io/badge/LinkedIn-Connect-blue?logo=linkedin" />
</a>
[![Portfolio](https://img.shields.io/badge/Portfolio-Visit-orange?logo=google-chrome)](https://siddharamayya.in)




> **Part 2 of the LangChain Tutorial Series.**
> Build a complete Retrieval-Augmented Generation (RAG) pipeline over real legal
> and regulatory documents — using only free, locally-run components for embeddings.

---

## What You Will Build

A working RAG system that:

1. Downloads real public regulatory PDFs (EU AI Act, NIST AI RMF)
2. Extracts and chunks text page-by-page with metadata
3. Embeds chunks using a free local HuggingFace model (no API cost)
4. Stores and searches vectors using FAISS
5. Retrieves the most relevant passages for any question
6. Feeds those passages to Gemini and gets a grounded, cited answer


```
PDF Documents
     ↓  load
Raw pages + metadata
     ↓  chunk
Overlapping text chunks
     ↓  embed (HuggingFace MiniLM, free)
384-dim vectors
     ↓  index
FAISS vector store (saved to disk)
     ↓  query time
User question → embed → search → top-5 chunks → Gemini → cited answer
```

---

## Why RAG?

LLMs are trained on a fixed snapshot of the internet. They cannot know about:
- Your private documents
- Recent regulations or news
- Internal company policies

They also **hallucinate** — they confidently generate plausible but wrong answers
when they don't know something.

RAG solves both by retrieving real text from your documents and injecting it into
the prompt, so the LLM answers from *your data*, not from memory.

---

## Setup

In [1]:
!pip install -U -q langchain langchain-community langchain-google-genai \
            langchain-text-splitters faiss-cpu pypdf \
            sentence-transformers google-generativeai requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
from google.colab import userdata
import os
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

## Part 6 — Document Loading

LangChain has loaders for PDFs, web pages, CSV, YouTube, Notion, GitHub, and more.
All return `Document` objects with `page_content` and `metadata`.

**Download real regulatory documents:**

In [4]:
import requests, os

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

DOCUMENTS = {
    "nist_ai_rmf.pdf":     "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf",
    "nist_genai_profile.pdf": "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.600-1.pdf",
}

os.makedirs("docs", exist_ok=True)

for filename, url in DOCUMENTS.items():
    path = f"docs/{filename}"
    if os.path.exists(path):
        print(f"✓ {filename} already exists")
        continue
    print(f"Downloading {filename}...")
    resp = requests.get(url, headers=HEADERS, timeout=90, stream=True)
    resp.raise_for_status()
    with open(path, "wb") as f:
        for chunk in resp.iter_content(8192):
            f.write(chunk)
    print(f"  ✓ {os.path.getsize(path)//1024} KB")

  ✓ 1900 KB
  ✓ 1147 KB



> **Why the custom User-Agent?** Government servers block Python's default `urllib`
> user-agent. Adding a browser user-agent string resolves 403 errors instantly.

**Load with LangChain:**

In [5]:
from langchain_community.document_loaders import PyPDFLoader

all_pages = []
for filename in os.listdir("docs"):
    if not filename.endswith(".pdf"):
        continue
    pages = PyPDFLoader(f"docs/{filename}").load()
    all_pages.extend(pages)
    print(f"{filename}: {len(pages)} pages")

print(f"\nTotal: {len(all_pages)} pages")
print(f"\nSample page content:\n{all_pages[2].page_content[:400]}")
print(f"\nMetadata: {all_pages[2].metadata}")
# → {"source": "docs/nist_ai_rmf.pdf", "page": 2}

nist_ai_rmf.pdf: 48 pages
nist_genai_profile.pdf: 64 pages

Total: 112 pages

Sample page content:
Certain commercial entities, equipment, or materials may be identified in this document in order to describe 
an experimental procedure or concept adequately. Such identification is not intended to imply recommenda-
tion or endorsement by the National Institute of Standards and Technology, nor is it intended to imply that 
the entities, materials, or equipment are necessarily the best available fo

Metadata: {'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-01-24T14:45:46-05:00', 'author': 'National Institute of Standards and Technology', 'keywords': 'Artificial Intelligence (AI); AI; AI RMF; AI RMF 1.0; AI systems; trustworthy and responsible AI.', 'moddate': '2025-06-04T13:01:45-04:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': 'As directed by the National Artificial Intell

**Other loaders you can swap in:**

In [7]:
!pip install -U -q Wikipedia

  Preparing metadata (setup.py) ... done


In [8]:
from langchain_community.document_loaders import (
    WebBaseLoader,      # any web page
    WikipediaLoader,    # Wikipedia directly
    CSVLoader,          # CSV files
    TextLoader,         # plain .txt files
)

# Wikipedia
docs = WikipediaLoader(query="EU AI Act", load_max_docs=2).load()

# Web page
docs = WebBaseLoader("https://example.com/policy").load()

## Part 7 — Text Splitting

You cannot embed an entire document as one vector — the meaning gets
averaged and diluted. Splitting into smaller chunks lets each chunk
have a precise, searchable meaning.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # ~200–250 words per chunk
    chunk_overlap=150,     # 150 chars shared between adjacent chunks
    separators=["\n\n", "\n", ".", " "]  # tries each in order — paragraph first
)

chunks = splitter.split_documents(all_pages)

print(f"Pages:  {len(all_pages)}")
print(f"Chunks: {len(chunks)}")
print(f"Avg chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")

print(f"\nSample chunk:\n{chunks[10].page_content}")
print(f"\nMetadata: {chunks[10].metadata}")

Pages:  112
Chunks: 350
Avg chunk length: 833 chars

Sample chunk:
to failures when they occur. AI systems are inherently socio-technical in nature, meaning
they are influenced by societal dynamics and human behavior. AI risks – and benefits –
can emerge from the interplay of technical aspects combined with societal factors related
to how a system is used, its interactions with other AI systems, who operates it, and the
social context in which it is deployed.
These risks make AI a uniquely challenging technology to deploy and utilize both for orga-
nizations and within society. Without proper controls, AI systems can amplify, perpetuate,
or exacerbate inequitable or undesirable outcomes for individuals and communities. With
proper controls, AI systems can mitigate and manage inequitable outcomes.
AI risk management is a key component of responsible development and use of AI sys-
tems. Responsible AI practices can help align the decisions about AI system design, de-
velopment, and uses 

**Why `chunk_overlap=150`?**

```
... Article 9 requires risk assessment. The assessment must cover all phases ...
                                       ↑
                        chunk 1 ends ──┤── chunk 2 begins
                                       │ ← 150 chars appear in BOTH chunks
```

Without overlap, a sentence split across a boundary would never be fully
represented in any single chunk, making it invisible to retrieval.

**Splitter comparison:**

| Splitter | Best for |
|---|---|
| `RecursiveCharacterTextSplitter` | General text — recommended default |
| `MarkdownTextSplitter` | Markdown files — splits on headers |
| `TokenTextSplitter` | When you need precise token count control |
| `RecursiveCharacterTextSplitter.from_language(Language.PYTHON)` | Source code |


---

## Part 8 — Embeddings and Vector Store

**What is an embedding?**

A list of numbers (a vector) that represents the *meaning* of text.
Similar meanings → similar vectors → they appear close together in vector space.

```
"AI provider obligations" → [0.23, -0.81, 0.44, ...]   (384 numbers)
"Duties of AI system providers" → [0.21, -0.79, 0.47, ...]  ← similar!
"The weather is nice" → [-0.55, 0.33, -0.12, ...]            ← far away
```

We use a **free local HuggingFace model** — no API calls, no cost, runs on CPU.

In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Downloads once (~90MB), cached after first run
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Build vector index from all chunks
print(f"Embedding {len(chunks)} chunks... (takes ~1–2 min on CPU)")
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save to disk — don't rebuild every session
vectorstore.save_local("legal_docs_index")
print("✓ Index saved to ./legal_docs_index")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 350 chunks... (takes ~1–2 min on CPU)
✓ Index saved to ./legal_docs_index


**Load from disk on subsequent runs:**

In [12]:
vectorstore = FAISS.load_local(
    "legal_docs_index",
    embeddings,
    allow_dangerous_deserialization=True   # required when loading from disk
)
print(f"✓ Loaded index")

✓ Loaded index


**Embedding model comparison:**

| Model | Dims | Size | Notes |
|---|---|---|---|
| `all-MiniLM-L6-v2` ← *we use* | 384 | ~90MB | Fast, free, good quality |
| `all-mpnet-base-v2` | 768 | ~420MB | Better quality, slower |
| `BAAI/bge-large-en-v1.5` | 1024 | ~1.3GB | Best open-source quality |
| Google `models/text-embedding-004` | 768 | API call | Excellent, uses Gemini API quota |

---

## Part 8b — Retrieval

Turn the vector store into a **retriever** — a standard LangChain interface
that can slot into any chain.

In [13]:
retriever = vectorstore.as_retriever(
    search_type="similarity",   # or "mmr" — more diverse results
    search_kwargs={"k": 5}      # return top 5 most relevant chunks
)

# Test it directly
question = "What are the obligations for providers of high-risk AI systems?"
results  = retriever.invoke(question)

for i, doc in enumerate(results):
    print(f"\n[{i+1}] Source: {doc.metadata['source']} | Page: {doc.metadata['page']}")
    print(doc.page_content[:300])


[1] Source: docs/nist_ai_rmf.pdf | Page: 5
to failures when they occur. AI systems are inherently socio-technical in nature, meaning
they are influenced by societal dynamics and human behavior. AI risks – and benefits –
can emerge from the interplay of technical aspects combined with societal factors related
to how a system is used, its inte

[2] Source: docs/nist_ai_rmf.pdf | Page: 9
NIST AI 100-1 AI RMF 1.0
Fig. 1. Examples of potential harms related to AI systems. Trustworthy AI systems and their
responsible use can mitigate negative risks and contribute to benefits for people, organizations, and
ecosystems.
1.2 Challenges for AI Risk Management
Several challenges are describe

[3] Source: docs/nist_ai_rmf.pdf | Page: 8
uals, communities, and society), organizations, and systems/ecosystems. Risk management
can enable AI developers and users to understand impacts and account for the inherent lim-
itations and uncertainties in their models and systems, which in turn can improve overa


**Search types:**

| Type | How it works | When to use |
|---|---|---|
| `similarity` | Pure cosine similarity | Default — fastest |
| `mmr` | Maximizes relevance AND diversity | When top results are too repetitive |

**Score threshold (optional — filter weak matches):**

In [14]:
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.6, "k": 5}
)

## Part 9 — Full RAG Chain

Combine everything: retrieval + prompt + LLM = grounded, cited answers.

In [24]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Format retrieved docs for the prompt
def format_docs(docs):
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')} | "
        f"Page: {doc.metadata.get('page', '?')}]\n{doc.page_content}"
        for doc in docs
    )

# Strict grounding prompt — forces LLM to use context only
rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant for legal and regulatory documents.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I cannot find this in the documents."
Always cite which source and page your answer comes from.

Context:
{context}

Question: {question}

Answer:
""")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# The full chain
rag_chain = (
    {
        "context":  retriever | format_docs,   # retriever gets question, formats docs
        "question": RunnablePassthrough()       # question passes through unchanged
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Ask it
answer = rag_chain.invoke("What does the GOVERN function cover in the NIST AI RMF?")
print(answer)

The GOVERN function in the NIST AI RMF covers the cultivation and implementation of a culture of risk management within organizations. It establishes the foundational structures, systems, processes, and teams necessary for managing AI risks. This includes:

*   Putting in place transparent and effectively implemented policies, processes, procedures, and practices across the organization related to mapping, measuring, and managing AI risks, including legal and regulatory requirements.
*   Ensuring ongoing monitoring and periodic review of the risk management process and its outcomes, with clearly defined organizational roles and responsibilities.
*   Establishing mechanisms to inventory AI systems, resourced according to organizational risk priorities.
*   Developing processes and procedures for safely decommissioning and phasing out AI systems without increasing risks or decreasing trustworthiness.
*   Implementing accountability structures so that appropriate teams and individuals are

**What `RunnablePassthrough()` does:**

In [21]:
from langchain_core.runnables import RunnablePassthrough

format_docs = {}
# This dict is actually a RunnableParallel under the hood
{
    "context":  retriever | format_docs,   # run retriever, format output
    "question": RunnablePassthrough()      # pass the input question through as-is
}
# Both run with the same input (the question string)
# Their outputs merge: {"context": "...", "question": "..."}
# That dict flows into rag_prompt

{'context': VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f6723aace00>, search_type='similarity_score_threshold', search_kwargs={'score_threshold': 0.6, 'k': 5})
 | {
     
   },
 'question': RunnablePassthrough()}

**Why `temperature=0` for RAG?**

RAG is a factual retrieval task, not a creative one. Temperature 0 makes
the LLM deterministic — it answers from the context without embellishment
or creative guessing. Higher temperature would introduce hallucination risk.

---

## Evaluating Your RAG Pipeline

A RAG system can fail in two independent ways:

| Layer | Failure | Symptom |
|---|---|---|
| Retrieval | Wrong chunks returned | Answer misses key facts that exist in the doc |
| Generation | LLM ignores context | Answer uses outside knowledge, not the doc |

Build a small golden test set to measure retrieval separately from generation:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [23]:
golden = [
    {
        "question": "What are the NIST AI RMF core functions?",
        "expected_source": "nist_ai_rmf.pdf",
        "expected_keywords": ["GOVERN", "MAP", "MEASURE", "MANAGE"]
    },
]

for item in golden:
    results = retriever.invoke(item["question"])
    sources  = [r.metadata.get("source", "") for r in results]
    text     = " ".join(r.page_content for r in results).lower()

    source_hit  = any(item["expected_source"] in s for s in sources)
    keyword_hits = sum(1 for kw in item["expected_keywords"] if kw.lower() in text)

    print(f"Source hit:   {'✓' if source_hit else '✗'}")
    print(f"Keywords:     {keyword_hits}/{len(item['expected_keywords'])}")

Source hit:   ✓
Keywords:     4/4


## Common Errors in This Notebook

| Error | Cause | Fix |
|---|---|---|
| `HTTP Error 403` on PDF download | Server blocks `urllib` default user-agent | Use `requests` with a browser User-Agent string |
| `ModuleNotFoundError: pypdf` | Not installed | `pip install pypdf` |
| `allow_dangerous_deserialization` | Loading FAISS index from disk | Add `allow_dangerous_deserialization=True` |
| Empty pages in loaded PDF | Image-only / scanned PDF | Switch to `pytesseract` OCR |
| Garbled text from PDF | Complex font encoding | Switch from `pypdf` to `pymupdf` (`pip install pymupdf`) |
| Low retrieval quality | Chunk size too large — signal diluted | Reduce to 500–800 chars |

---

## Quick Reference

In [26]:
# Load PDF
from langchain_community.document_loaders import PyPDFLoader
pages = PyPDFLoader("docs/nist_ai_rmf.pdf").load()

# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(pages)

# Embed + Store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
embeddings   = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore  = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("my_index")

# Load + Retrieve
vectorstore = FAISS.load_local("my_index", embeddings, allow_dangerous_deserialization=True)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 5})
docs        = retriever.invoke("your question")

# RAG Chain
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)
answer = rag_chain.invoke("What are the NIST AI RMF core functions?")

print("Answer")
print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer
The NIST AI RMF Core is composed of four functions: GOVERN, MAP, MEASURE, and MANAGE. (Source: docs/nist_ai_rmf.pdf | Page: 7, 24)


## What's Next

**[→ Notebook 03: Agents](./03_agents.md)**
Streaming responses token-by-token, and building LLM agents that autonomously
use tools like Wikipedia and web search to answer questions.

---

*Part of the [LangChain Tutorial Series](../README.md) — built with LangChain 0.3.x and Google Gemini 1.5 Flash*
